In [1]:
import os
import psutil
import gc
from pypdf import PdfReader
from tqdm import tqdm

def check_memory():
    # Memory used by this specific notebook process
    process = psutil.Process(os.getpid())
    process_mem = process.memory_info().rss / (1024 * 1024)
    
    # Total system available memory
    virtual_mem = psutil.virtual_memory()
    available_mb = virtual_mem.available / (1024 * 1024)
    
    print(f"--- 📊 Memory Dashboard ---")
    print(f"Notebook Process: {process_mem:.2f} MB")
    print(f"System Available: {available_mb:.2f} MB")
    print(f"Total Load: {virtual_mem.percent}%")
    print(f"---------------------------")

check_memory()

--- 📊 Memory Dashboard ---
Notebook Process: 70.51 MB
System Available: 565.54 MB
Total Load: 92.4%
---------------------------


In [2]:
# Extraction
# --------------

pdf_name = "alphabet_10k_2025.pdf"  # Double-check this matches your filename exactly

def safe_extract_and_chunk(filename, size=500, overlap=50):
    if not os.path.exists(filename):
        print(f"❌ Error: {filename} not found in this folder!")
        return None
    
    print(f"📖 Opening {filename}...")
    reader = PdfReader(filename)
    all_chunks = []
    current_text = ""

    # Processing page-by-page to avoid one giant string in RAM
    for page in tqdm(reader.pages, desc="Extracting Text"):
        page_text = page.extract_text() or ""
        current_text += page_text + " "
        
        # While the buffer is larger than our chunk size, 
        # carve off chunks and keep the overlap remainder
        while len(current_text) > size:
            chunk = current_text[:size]
            all_chunks.append(chunk)
            current_text = current_text[size - overlap:]
            
    # Capture the final bit of text
    if current_text:
        all_chunks.append(current_text)

    print(f"\n✅ Created {len(all_chunks)} text chunks.")
    
    # Force immediate memory cleanup
    del reader
    gc.collect() 
    return all_chunks

# Run the extraction
document_chunks = safe_extract_and_chunk(pdf_name)

# Final check of the 'heartbeat'
if document_chunks:
    check_memory()

📖 Opening alphabet_10k_2025.pdf...


Extracting Text: 100%|██████████| 99/99 [00:13<00:00,  7.40it/s]


✅ Created 776 text chunks.
--- 📊 Memory Dashboard ---
Notebook Process: 78.84 MB
System Available: 1099.59 MB
Total Load: 85.2%
---------------------------


In [3]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. Load the model (~80-100MB in RAM)
print("📥 Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Encode the chunks (The actual 'thinking' part)
print(f"🔄 Encoding {len(document_chunks)} chunks. Your Ryzen CPU is working...")
# We use convert_to_numpy=True because FAISS needs numpy arrays
embeddings = model.encode(document_chunks, show_progress_bar=True, convert_to_numpy=True)

# 3. Create the FAISS Index (The Search Engine)
dimension = embeddings.shape[1] # This is 384 for this specific model
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print("✅ FAISS Index successfully built!")

# 4. Memory Cleanup
# We keep the 'model' and 'index', but we can delete the raw 'embeddings' 
# list because they are now stored inside the 'index'.
del embeddings
gc.collect()

check_memory()

d:\Projects\LLM_RAG_Projects\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading embedding model (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1768.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Encoding 776 chunks. Your Ryzen CPU is working...


Batches: 100%|██████████| 25/25 [00:31<00:00,  1.28s/it]


✅ FAISS Index successfully built!
--- 📊 Memory Dashboard ---
Notebook Process: 619.19 MB
System Available: 1241.83 MB
Total Load: 83.3%
---------------------------


In [4]:
# Test the search functionality of the vector database with a sample question

def ask_question(query, k=3):
    # 1. Turn the question into a vector
    query_vector = model.encode([query]).astype('float32')
    
    # 2. Search FAISS for the 'k' nearest neighbors
    # 'D' is distances, 'I' is the indices of the chunks
    D, I = index.search(query_vector, k)
    
    print(f"🔎 Query: {query}\n")
    print(f"--- 📄 Top {k} Relevant Chunks ---")
    
    for i in range(len(I[0])):
        chunk_idx = I[0][i]
        distance = D[0][i]
        print(f"\n[Result {i+1}] (Distance: {distance:.4f})")
        print(f"{document_chunks[chunk_idx]}")
        print("-" * 30)

# Try a specific question about the 2025 data
ask_question("What are the main risks mentioned regarding AI and Gemini?")

🔎 Query: What are the main risks mentioned regarding AI and Gemini?

--- 📄 Top 3 Relevant Chunks ---

[Result 1] (Distance: 0.9983)
forms. We continue to help our users  access information and knowledge, express themselves, and get things 
done by embedding the power of generative AI and Gemini into our products and platforms.  Today, all 15 of our half-
billion-user products — including seven with two billion users — use our Gemini models. For our Google Cloud 
customers, our offerings are helping organizations stay at the forefront of innovation with solutions such as Gemini 
Enterprise and Gemini for Google Workspace. 
Gu
------------------------------

[Result 2] (Distance: 0.9983)
and the market's understanding of AI-centric security risks and protection 
methods continue to develop. We have in the past discovered, and may in the future discover, some errors in our 
software code only after we have released the code. Systems and control failures, security breaches, failure to comp